# TTA-UC現象のGKSL-Lindblad量子ダイナミクス：包括的比較

このノートブックでは、TTA-UC（Triplet-Triplet Annihilation Upconversion）現象の
GKSL-Lindblad量子ダイナミクスシミュレーションを6つのシナリオで実行し、結果を比較します。

## シナリオ一覧

| シナリオ | 手法 | ボソン | シミュレータ |
|----------|------|--------|------------|
| 1 | 古典ODE (RK45) | なし | ClassicalGKSLSimulator |
| 2 | 古典ODE (BDF) | あり | ClassicalGKSLBosonSimulator |
| 3 | Qubit Stinespring+Trotter | なし | QubitGKSLSimulator |
| 4 | Qubit Stinespring+Trotter | あり | QubitGKSLBosonSimulator |
| 5 | Qudit Stinespring+Trotter | なし | QuditGKSLSimulator |
| 6 | Qudit Stinespring+Trotter | あり | QuditGKSLBosonSimulator |

## GKSL-Lindblad方程式

$$\frac{d\hat{\rho}}{dt} = -\frac{i}{\hbar}[\hat{H}, \hat{\rho}] + \sum_\alpha \left( \hat{L}_\alpha \hat{\rho} \hat{L}_\alpha^\dagger - \frac{1}{2}\{\hat{L}_\alpha^\dagger \hat{L}_\alpha, \hat{\rho}\}\right)$$

## 1. 共通パラメータの設定

In [ ]:
import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from gksl_physical_parameters import GKSLPhysicalParameters

# Non-boson parameters (4 molecules, d=3, dim=81)
params = GKSLPhysicalParameters(
    E_T=1.5,  # Triplet energy (eV)
    E_S=3.0,  # Singlet energy (eV)
    V=0.1,  # Transfer coupling (eV)
    gamma_TTA=0.05,  # TTA rate
    Gamma_fl=0.01,  # Fluorescence rate
    Gamma_ph=1e-6,  # Phosphorescence rate
    k_IC=0.005,  # Internal conversion rate
    k_ISC_ST=0.003,  # Intersystem crossing S->T
    k_ISC_TS=1e-5,  # Intersystem crossing T->S
)

# Boson parameters (2 molecules for tractable computation)
params_boson = GKSLPhysicalParameters(
    N_molecules=2,
    with_boson=True,
    n_max=1,
    omega_ph=0.15,
    g_eph=0.02,
)

# Simulation settings
t_max = 100.0
n_steps = 100

# For quantum simulators (Stinespring+Trotter),
# more steps are needed for accuracy but increase computation time.
# Adjust n_steps_quantum as needed.
n_steps_quantum = 100

## 2. シナリオ1: 古典GKSL（ボソン無し）

scipy.integrate.solve_ivp (RK45) による直接密度行列ODE積分。
81×81次元の密度行列を直接積分します。

In [ ]:
from classical_gksl_simulator import ClassicalGKSLSimulator
from gksl_visualization import plot_entropy_and_purity, plot_population_dynamics

sim1 = ClassicalGKSLSimulator(params)
result1 = sim1.simulate(t_max=t_max, n_steps=n_steps, initial_state="edge_triplet")


plot_population_dynamics(result1, title="Scenario 1: Classical GKSL (No Boson)")
plot_entropy_and_purity(result1, title="Scenario 1: Entropy & Purity")

## 3. シナリオ5: Qudit GKSL（ボソン無し）

ネイティブqutrit(d=3) + Stinespring dilation + 2次対称Trotter分解。
4 qutrit + 26 ancilla qubit = 30量子ビット相当。
禁止状態なし（qutritの自然なエンコーディング）。

In [ ]:
from qudit_gksl_simulator import QuditGKSLSimulator

sim5 = QuditGKSLSimulator(params)
result5 = sim5.simulate(t_max=t_max, n_steps=n_steps_quantum, initial_state="edge_triplet")


plot_population_dynamics(result5, title="Scenario 5: Qudit GKSL (No Boson)")

## 4. シナリオ3: Qubit GKSL（ボソン無し）

2-qubitエンコーディング(|00⟩=S0, |01⟩=T1, |10⟩=S1, |11⟩=禁止) + Stinespring + Trotter。
8 system qubit + 26 ancilla = 34 qubit。

In [ ]:
from qubit_gksl_simulator import QubitGKSLSimulator

sim3 = QubitGKSLSimulator(params)
result3 = sim3.simulate(t_max=t_max, n_steps=n_steps_quantum, initial_state="edge_triplet")


plot_population_dynamics(result3, title="Scenario 3: Qubit GKSL (No Boson)")

## 5. シナリオ2: 古典GKSL（ボソン有り）

Holstein型電子-フォノン結合を含むBDF法ODE積分。
計算負荷のため、2分子・n_max=1で実行（dim=36）。

In [ ]:
from classical_gksl_boson_simulator import ClassicalGKSLBosonSimulator

sim2 = ClassicalGKSLBosonSimulator(params_boson)
result2 = sim2.simulate(t_max=t_max, n_steps=n_steps, initial_state="edge_triplet")


plot_population_dynamics(result2, title="Scenario 2: Classical GKSL (With Boson, N=2)")

## 6. シナリオ6: Qudit GKSL（ボソン有り）

qutrit電子系 + qutritフォノン系 + Stinespring dilation。
2分子・n_max=1で実行。

In [ ]:
from qudit_gksl_boson_simulator import QuditGKSLBosonSimulator

sim6 = QuditGKSLBosonSimulator(params_boson)
result6 = sim6.simulate(t_max=t_max, n_steps=n_steps_quantum, initial_state="edge_triplet")


plot_population_dynamics(result6, title="Scenario 6: Qudit GKSL (With Boson, N=2)")

## 7. シナリオ4: Qubit GKSL（ボソン有り）

Qubit Stinespring + Trotter in 拡張空間。
2分子・n_max=1で実行。

In [ ]:
from qubit_gksl_boson_simulator import QubitGKSLBosonSimulator

sim4 = QubitGKSLBosonSimulator(params_boson)
result4 = sim4.simulate(t_max=t_max, n_steps=n_steps_quantum, initial_state="edge_triplet")


plot_population_dynamics(result4, title="Scenario 4: Qubit GKSL (With Boson, N=2)")

## 8. 全シナリオの包括的比較

ボソン無しの3シナリオ（同一パラメータ）とボソン有りの3シナリオ（同一パラメータ）を
それぞれ比較します。

In [ ]:
from gksl_visualization import compare_multiple_scenarios

# Non-boson comparison (all use same 81-dim Hilbert space)
results_nb = {
    "Classical": result1,
    "Qubit": result3,
    "Qudit": result5,
}
compare_multiple_scenarios(results_nb, title="Non-Boson Scenarios Comparison (N=4)")

# Boson comparison (N=2, n_max=1)
results_b = {
    "Classical": result2,
    "Qubit": result4,
    "Qudit": result6,
}
compare_multiple_scenarios(results_b, title="Boson Scenarios Comparison (N=2, n_max=1)")

## 9. ユニタリ vs GKSL-Lindblad 比較

全散逸率を0にした純ユニタリ発展（閉量子系）と、
散逸ありのGKSL-Lindblad発展（開量子系）を比較します。

In [ ]:
from gksl_visualization import plot_gksl_comparison

# Unitary limit (all dissipation rates = 0)
params_unitary = GKSLPhysicalParameters(
    gamma_TTA=0,
    Gamma_fl=0,
    Gamma_ph=0,
    k_IC=0,
    k_ISC_ST=0,
    k_ISC_TS=0,
)
sim_unitary = ClassicalGKSLSimulator(params_unitary)
result_unitary = sim_unitary.simulate(t_max=t_max, n_steps=n_steps, initial_state="edge_triplet")


plot_gksl_comparison(result_unitary, result1, title="Unitary vs GKSL-Lindblad")

## 10. 検証と考察

In [ ]:
from gksl_visualization import plot_trace_conservation

# Trace conservation check for all scenarios
for _name, result in [
    ("Classical NB", result1),
    ("Classical B", result2),
    ("Qubit NB", result3),
    ("Qubit B", result4),
    ("Qudit NB", result5),
    ("Qudit B", result6),
]:
    max_dev = max(abs(t - 1.0) for t in result["trace"])

for _name, result in [("Classical NB", result1), ("Qubit NB", result3), ("Qudit NB", result5)]:
    N_mol = 4
    max_dev = max(abs(p["N_S0"] + p["N_T1"] + p["N_S1"] - N_mol) for p in result["populations"])

for _name, result in [
    ("Classical NB", result1),
    ("Classical B", result2),
    ("Qubit NB", result3),
    ("Qubit B", result4),
    ("Qudit NB", result5),
    ("Qudit B", result6),
]:
    pass

# Trace conservation plot for classical reference
plot_trace_conservation(result1, title="Trace Conservation (Classical NB)")

## 11. Stinespring忠実度の評価

古典ODE結果（理想値）とStinespring+Trotter結果の量子忠実度を計算します。

$$F(\hat{\rho}, \hat{\sigma}) = \left(\mathrm{Tr}\sqrt{\sqrt{\hat{\rho}}\hat{\sigma}\sqrt{\hat{\rho}}}\right)^2$$

In [ ]:
def quantum_fidelity(rho, sigma):
    """Compute quantum state fidelity via eigendecomposition."""
    evals, evecs = np.linalg.eigh(rho)
    evals = np.maximum(evals, 0.0)
    sqrt_rho = evecs @ np.diag(np.sqrt(evals)) @ evecs.conj().T
    M = sqrt_rho @ sigma @ sqrt_rho
    M = (M + M.conj().T) / 2
    evals_M = np.linalg.eigvalsh(M)
    evals_M = np.maximum(evals_M, 0.0)
    return float(np.real(np.sum(np.sqrt(evals_M))) ** 2)


# Compare classical vs quantum final density matrices (non-boson)
F_qudit = quantum_fidelity(result1["rho_final"], result5["rho_final"])
F_qubit = quantum_fidelity(result1["rho_final"], result3["rho_final"])

## 12. まとめ

### 各シナリオの特徴

| シナリオ | Hilbert空間次元 | 量子資源 | 推定ゲート数/ステップ | 特徴 |
|----------|---------------|---------|---------------------|------|
| 1: Classical NB | 81 | - | - | 基準実装（厳密ODE積分） |
| 2: Classical B | 36 (N=2,n=1) | - | - | 電子-フォノン結合モデル |
| 3: Qubit NB | 81 | 34 qubits | ~194 | 2-qubitエンコード、禁止状態あり |
| 4: Qubit B | 拡張 | 42 qubits | ~194+ | ボソンモード含む |
| 5: Qudit NB | 81 | 4 qutrits + 26 ancilla | ~33 | ネイティブd=3、禁止状態なし |
| 6: Qudit B | 拡張 | 8 qutrits + 26 ancilla | ~33+ | qutritフォノンモード |

### 設計原則

- **ヒューリスティック・フォールバックの不使用**: 密度行列の強制正規化、固有値クリッピング等は一切不使用
- **厳密な物理的検証**: トレース保存、正定値性、粒子数保存を各ステップで検証
- **全エラーは例外として送出**: PhysicsViolationError, NumericalInstabilityError等